# TP 4 — Prototype de priorisation des événements de sécurité

**Équipe :** _(vos noms)_

Ce notebook est un point de départ. Il ne contient **aucune solution** : à vous de le remplir.

Rappel de la mission : concevoir et évaluer un prototype de classification répartissant les événements entre `NORMAL`, `SUSPICIOUS` et `MALICIOUS`.

> Votre objectif n'est pas uniquement d'obtenir le meilleur score statistique. Vous devez proposer un modèle cohérent avec les conséquences opérationnelles des faux positifs et des faux négatifs.

## 0. Imports

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

import matplotlib.pyplot as plt

RANDOM_STATE = 42  # à conserver pour que vos résultats soient reproductibles

## 1. Chargement des données

In [3]:
df = pd.read_csv("../data/tp4/cleaned_dataset.csv")
df.head()

,timestamp,source,event_id,user_id,device_id,src_ip,src_ip_valid,event_type,process_name,severity,...,employment_status,privileged_account,hostname,asset_type,asset_owner,asset_criticality,user_known,device_known,label,triage_label
0,2026-03-02 01:31:00,authentication_logs,AUTH003692,U0190,D0154,10.27.255.130,True,MFA_CHALLENGE,UNKNOWN,MEDIUM,...,ACTIVE,YES,PHF-LT-0154,Mobile,Finance,HIGH,True,True,UNLABELED,SUSPICIOUS
1,2026-03-02 01:41:00,authentication_logs,AUTH000733,U0008,D0029,10.21.247.242,True,LOGOUT,UNKNOWN,LOW,...,ACTIVE,NO,PHF-WS-0029,Server,IT,CRITICAL,True,True,UNLABELED,MALICIOUS
2,2026-03-02 01:53:00,authentication_logs,AUTH000859,U0343,D0025,10.12.238.24,True,LOGIN,UNKNOWN,MEDIUM,...,ACTIVE,NO,PHF-WS-0025,Laptop,RH,LOW,True,True,UNLABELED,NORMAL
3,2026-03-02 02:32:15,authentication_logs,AUTH002474,U0202,D0053,10.42.152.134,True,PASSWORD_CHANGE,UNKNOWN,MEDIUM,...,ACTIVE,NO,PHF-SRV-0053,Server,Regions,MEDIUM,True,True,UNLABELED,SUSPICIOUS
4,2026-03-02 02:50:24,authentication_logs,AUTH002027,U0226,D0041,10.39.189.42,True,MFA_CHALLENGE,UNKNOWN,LOW,...,ACTIVE,NO,PHF-SRV-0041,VM,Direction,CRITICAL,True,True,UNLABELED,SUSPICIOUS


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6702 entries, 0 to 6701
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   timestamp              6702 non-null   str  
 1   source                 6702 non-null   str  
 2   event_id               6702 non-null   str  
 3   user_id                6702 non-null   str  
 4   device_id              6702 non-null   str  
 5   src_ip                 6702 non-null   str  
 6   src_ip_valid           6702 non-null   bool 
 7   event_type             6702 non-null   str  
 8   process_name           6702 non-null   str  
 9   severity               6702 non-null   str  
 10  authentication_result  6702 non-null   str  
 11  location               6702 non-null   str  
 12  user_department        6702 non-null   str  
 13  user_role              6702 non-null   str  
 14  employment_status      6702 non-null   str  
 15  privileged_account     6702 non-null   str  
 16 

## 2. Analyse de la cible

À traiter :
- distribution des trois classes ;
- ampleur du déséquilibre ;
- conséquence sur le choix des métriques.

**Question :** si un modèle prédisait toujours la classe majoritaire, quelle accuracy obtiendrait-il ?

In [21]:
# Nombre d'événements par classe
distribution = df["triage_label"].value_counts()

print("Distribution des classes :")
print(distribution)

print("\nRépartition en pourcentage :")
print(
    (distribution / len(df) * 100)
    .round(2)
)

Distribution des classes :
triage_label
NORMAL        4948
SUSPICIOUS    1284
MALICIOUS      470
Name: count, dtype: int64

Répartition en pourcentage :
triage_label
NORMAL        73.83
SUSPICIOUS    19.16
MALICIOUS      7.01
Name: count, dtype: float64


### Analyse

La classe `NORMAL` est largement majoritaire avec environ 73,8 % des événements.
La classe `MALICIOUS` ne représente qu'environ 7 % du dataset.

Le dataset est donc déséquilibré. L'accuracy seule ne sera pas suffisante pour
évaluer correctement les modèles, car elle peut masquer de mauvaises performances
sur les classes minoritaires.

Il faudra notamment étudier le recall et le F1-score de chaque classe,
en particulier pour `MALICIOUS`.

## 3. Colonnes utilisables et fuite de cible

Passez en revue **chaque colonne** et décidez si elle peut servir de variable d'entrée.

Pour chacune, posez-vous la question : *cette information est-elle disponible au moment où le système doit prioriser un événement qui vient d'arriver ?*

Documentez vos exclusions et leur motif — c'est un livrable attendu.

In [20]:
# TODO : lister les colonnes, décider lesquelles exclure et pourquoi

# Liste des colonnes du dataset
print(df.columns.tolist())

# Colonnes à exclure du modèle
excluded_columns = [
    "triage_label",  # cible à prédire
    "label",         # décision analyste disponible après investigation
    "event_id"       # identifiant unique, pas une information métier utile
]

print("\nColonnes exclues :")
for col in excluded_columns:
    print("-", col)

print("\nLien entre label analyste et triage_label :")

print(
    pd.crosstab(
        df["label"],
        df["triage_label"],
        normalize="index"
    ).round(3)
)

['timestamp', 'source', 'event_id', 'user_id', 'device_id', 'src_ip', 'src_ip_valid', 'event_type', 'process_name', 'severity', 'authentication_result', 'location', 'user_department', 'user_role', 'employment_status', 'privileged_account', 'hostname', 'asset_type', 'asset_owner', 'asset_criticality', 'user_known', 'device_known', 'label', 'triage_label']

Colonnes exclues :
- triage_label
- label
- event_id

Lien entre label analyste et triage_label :
triage_label         MALICIOUS  NORMAL  SUSPICIOUS
label                                             
FALSE_POSITIVE           0.036   0.810       0.155
NEEDS_INVESTIGATION      0.789   0.025       0.186
TRUE_POSITIVE            1.000   0.000       0.000
UNLABELED                0.045   0.751       0.204


### Fuite de cible

La colonne `label` ne doit pas être utilisée comme variable d'entrée.

Elle correspond à la décision prise par l'analyste après le traitement de
l'événement. Or le modèle doit prioriser l'événement avant cette investigation.

L'utilisation de cette colonne constituerait donc une fuite de cible :
le modèle disposerait indirectement d'une information sur la réponse qu'il
cherche à prédire.

Le lien est particulièrement fort puisque les événements `TRUE_POSITIVE`
sont tous associés à la classe `MALICIOUS`.

`triage_label` est également exclue des variables d'entrée puisqu'il s'agit
de la cible à prédire. `event_id` est exclu car il s'agit uniquement d'un
identifiant unique sans valeur prédictive métier.

## 4. Feature engineering

Certains signaux utiles ne sont pas dans le fichier et doivent être construits.

Pistes (ni obligatoires ni exhaustives) : heure de l'événement, activité nocturne, échecs d'authentification récents pour un même utilisateur, succès après une série d'échecs, volume d'activité par utilisateur ou machine, localisation inhabituelle, croisement privilèges × criticité de l'actif.

Justifiez les variables que vous retenez.

In [22]:
# TODO : construire vos features

# Copie du dataset pour construire les variables du modèle
df_model = df.copy()

# Conversion du timestamp en vraie date
df_model["timestamp"] = pd.to_datetime(df_model["timestamp"])

# Informations temporelles
df_model["hour"] = df_model["timestamp"].dt.hour
df_model["day_of_week"] = df_model["timestamp"].dt.dayofweek

# Événement réalisé la nuit : entre 22h et 6h
df_model["is_night"] = (
    (df_model["hour"] >= 22)
    | (df_model["hour"] < 6)
).astype(int)

# Échec d'authentification
df_model["failed_auth"] = (
    df_model["authentication_result"] == "FAILED"
).astype(int)

# Combinaison compte privilégié + actif critique
df_model["privileged_critical"] = (
    (df_model["privileged_account"] == "YES")
    & (df_model["asset_criticality"] == "CRITICAL")
).astype(int)

print(
    df_model[
        [
            "timestamp",
            "hour",
            "day_of_week",
            "is_night",
            "failed_auth",
            "privileged_critical"
        ]
    ].head()
)

            timestamp  hour  day_of_week  is_night  failed_auth  \
0 2026-03-02 01:31:00     1            0         1            0   
1 2026-03-02 01:41:00     1            0         1            1   
2 2026-03-02 01:53:00     1            0         1            0   
3 2026-03-02 02:32:15     2            0         1            0   
4 2026-03-02 02:50:24     2            0         1            0   

   privileged_critical  
0                    0  
1                    0  
2                    0  
3                    0  
4                    0  


### Features retenues

Plusieurs variables simples ont été construites à partir des informations
disponibles au moment de l'événement :

- `hour` : heure de l'événement ;
- `day_of_week` : jour de la semaine ;
- `is_night` : indique si l'événement se produit entre 22h et 6h ;
- `failed_auth` : indique un échec d'authentification ;
- `privileged_critical` : combinaison entre un compte privilégié et un actif critique.

Ces variables ont été retenues car elles peuvent apporter du contexte au modèle
sans utiliser d'information issue de l'investigation humaine.

Les variables temporelles plus complexes comme le nombre d'échecs récents ou
la fréquence d'activité pourront être ajoutées ensuite, mais elles doivent être
calculées uniquement à partir des événements antérieurs afin d'éviter une fuite
d'information.

## 5. Séparation train / test

Pensez à la **stratification** : avec une classe minoritaire à ~7 %, un split non stratifié peut déséquilibrer fortement le jeu de test.

In [ ]:
# TODO : train_test_split stratifié
# Cible à prédire
y = df_model["triage_label"]

# Variables d'entrée
X = df_model.drop(
    columns=[
        "triage_label",
        "label",
        "event_id",
        "timestamp",
        "user_id",
        "device_id",
        "src_ip",
        "hostname"
    ]
)

# Séparation train / test en conservant la proportion des classes
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Taille du train :", len(X_train))
print("Taille du test  :", len(X_test))

print("\nDistribution dans le train :")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nDistribution dans le test :")
print((y_test.value_counts(normalize=True) * 100).round(2))

Taille du train : 5361
Taille du test  : 1341

Distribution dans le train :
triage_label
NORMAL        73.83
SUSPICIOUS    19.16
MALICIOUS      7.01
Name: proportion, dtype: float64

Distribution dans le test :
triage_label
NORMAL        73.83
SUSPICIOUS    19.16
MALICIOUS      7.01
Name: proportion, dtype: float64


### Séparation des données

Le dataset est séparé en 80 % pour l'entraînement et 20 % pour le test.

La séparation est stratifiée sur `triage_label` afin de conserver une proportion
similaire de `NORMAL`, `SUSPICIOUS` et `MALICIOUS` dans les deux jeux.

Le jeu de test reste totalement séparé de l'entraînement et servira uniquement
à mesurer les performances finales des modèles.

## 6. Preprocessing

Les variables catégorielles doivent être encodées, les numériques mises à l'échelle pour certains modèles. Utilisez un `ColumnTransformer` dans un `Pipeline` : cela évite toute fuite entre train et test.

In [9]:
# TODO : ColumnTransformer + Pipeline


## 7. Baseline **obligatoire**

Avant tout modèle, construisez une baseline naïve : par exemple prédire systématiquement la classe majoritaire (`DummyClassifier`).

C'est le point de comparaison sans lequel vos scores ne veulent rien dire.

In [10]:
# TODO : baseline


## 8. Entraîner au moins deux modèles

Par exemple parmi : régression logistique, arbre de décision, forêt aléatoire.

Ne présumez pas qu'un modèle complexe sera meilleur : vérifiez-le.

In [11]:
# TODO : modèle 1


In [12]:
# TODO : modèle 2


## 9. Évaluation

Pour chaque modèle, produisez :
- la matrice de confusion (3×3) ;
- precision, recall et F1 **par classe** ;
- le macro F1 et le F1 pondéré ;
- le nombre de faux positifs et de faux négatifs.

Comparez systématiquement à la baseline.

In [13]:
# TODO : évaluation comparée


## 10. Analyse des erreurs et arbitrage métier

**Question centrale :** entre un faux positif et un faux négatif, lequel est le plus grave dans ce contexte ?

- Faux positif : un événement bénin est remonté aux analystes.
- Faux négatif : un événement malveillant n'est jamais investigué.

Quel compromis retenez-vous, et pourquoi ? Votre réponse doit être argumentée : il n'y a pas une seule bonne réponse, mais il y a des justifications solides et des justifications faibles.

In [14]:
# TODO : analyse des erreurs


## 11. Choix du modèle final

Quel modèle recommandez-vous à la direction, et sur quels critères ?

Attention : le meilleur modèle statistique n'est pas nécessairement le meilleur choix opérationnel.

In [15]:
# TODO


## 12. Impact projet

Dernière étape, et pas la moins importante.

Au cours de la séance, le formateur vous remettra des **cartes événement** décrivant des changements survenant sur le projet. Pour chacune, mettez à jour :

- le backlog ;
- le planning ;
- le Risk Register ;
- l'architecture si nécessaire.

Documentez chaque décision : impact, criticité, responsable, arbitrage retenu.

In [16]:
# TODO : suivi des décisions prises suite aux cartes événement
